In [1]:
from google.colab import files
uploaded = files.upload()


Saving diabetic_data.csv to diabetic_data.csv


In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('diabetic_data.csv')

print(f"Original dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

Original dataset: 101,766 rows × 50 columns


In [3]:
# Calculate percentage of missing values (including '?')
def calculate_missing_pct(df):
    missing_pct = {}
    for col in df.columns:
        # Count NaN
        nan_count = df[col].isnull().sum()
        # Count '?'
        question_count = (df[col] == '?').sum()
        # Total missing
        total_missing = nan_count + question_count
        missing_pct[col] = (total_missing / len(df)) * 100

    return pd.Series(missing_pct).sort_values(ascending=False)

# Calculate
missing_percentages = calculate_missing_pct(df)

# Show columns with >50% missing
print("Columns with >50% missing data:")
print("="*60)
high_missing = missing_percentages[missing_percentages > 50]
print(high_missing)

Columns with >50% missing data:
weight           96.858479
max_glu_serum    94.746772
A1Cresult        83.277322
dtype: float64


In [4]:
# Columns to drop
columns_to_drop = ['weight', 'max_glu_serum', 'A1Cresult']

# Drop them
df_clean = df.drop(columns=columns_to_drop)

print(f"Original shape: {df.shape}")
print(f"After dropping high-missing columns: {df_clean.shape}")
print(f"Columns removed: {len(columns_to_drop)}")

Original shape: (101766, 50)
After dropping high-missing columns: (101766, 47)
Columns removed: 3


In [5]:
# Replace all '?' with NaN
df_clean = df_clean.replace('?', np.nan)

# Check how many missing values we have now
print("Columns with missing values after converting '?':")
print("="*60)
missing = df_clean.isnull().sum()
missing_with_values = missing[missing > 0].sort_values(ascending=False)
print(missing_with_values)

Columns with missing values after converting '?':
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64


In [6]:
# Drop high-missing columns
columns_to_drop_2 = ['medical_specialty', 'payer_code']

df_clean = df_clean.drop(columns=columns_to_drop_2)

print(f"After dropping medical_specialty and payer_code:")
print(f"Shape: {df_clean.shape}")
print(f"Total columns dropped so far: {3 + len(columns_to_drop_2)}")

After dropping medical_specialty and payer_code:
Shape: (101766, 45)
Total columns dropped so far: 5


In [7]:
# Fill missing values with 'Unknown' for categorical columns
columns_to_fill = ['race', 'diag_1', 'diag_2', 'diag_3']

for col in columns_to_fill:
    df_clean[col] = df_clean[col].fillna('Unknown')
    print(f"Filled {col} missing values with 'Unknown'")

# Verify no missing values remain
print("\n" + "="*60)
print("Missing values after filling:")
print("="*60)
remaining_missing = df_clean.isnull().sum()
print(remaining_missing[remaining_missing > 0])

if remaining_missing.sum() == 0:
    print("\n SUCCESS! No missing values remaining!")
else:
    print(f"\n Still have {remaining_missing.sum()} missing values to handle")

Filled race missing values with 'Unknown'
Filled diag_1 missing values with 'Unknown'
Filled diag_2 missing values with 'Unknown'
Filled diag_3 missing values with 'Unknown'

Missing values after filling:
Series([], dtype: int64)

✅ SUCCESS! No missing values remaining!


In [8]:
# Check for duplicates
print("Checking for duplicate rows...")
print("="*60)

duplicates_count = df_clean.duplicated().sum()
print(f"Number of duplicate rows: {duplicates_count:,}")

if duplicates_count > 0:
    # Remove duplicates
    df_clean = df_clean.drop_duplicates()
    print(f" Removed {duplicates_count:,} duplicate rows")
    print(f"New shape: {df_clean.shape}")
else:
    print(" No duplicates found!")

print("="*60)

Checking for duplicate rows...
Number of duplicate rows: 0
 No duplicates found!


In [10]:
# Create binary target
df_clean['readmitted_30day'] = (df_clean['readmitted'] == '<30').astype(int)

# Quick check
print(df_clean['readmitted_30day'].value_counts())

readmitted_30day
0    90409
1    11357
Name: count, dtype: int64


In [11]:
# Save cleaned dataset
df_clean.to_csv('diabetic_data_cleaned.csv', index=False)

print("="*60)
print("DATA CLEANING COMPLETE!")
print("="*60)
print(f"Original dataset: {df.shape}")
print(f"Cleaned dataset: {df_clean.shape}")
print(f"Columns removed: {df.shape[1] - df_clean.shape[1]}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")
print(f"\n✅ Saved as: diabetic_data_cleaned.csv")
print("="*60)

DATA CLEANING COMPLETE!
Original dataset: (101766, 50)
Cleaned dataset: (101766, 46)
Columns removed: 4
Rows removed: 0

✅ Saved as: diabetic_data_cleaned.csv


In [12]:
# Download the cleaned dataset
from google.colab import files
files.download('diabetic_data_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>